# Protocol 7 — Build an interpretable classifier

Turn two labelled sets of sequences into a **working, interpretable predictor**. This protocol chains the standard AAanalysis pipeline — split sequences into *parts*, mine a *CPP signature*, build a *feature matrix*, and fit a `TreeModel` — into a classifier whose every input feature is a human-readable `PART-SPLIT-SCALE` feature id tied to an AAontology physicochemical property. You read out *which* determinants drive the prediction, not just a score.

It is the direct successor to **Protocol 1 — CPP signature**: Protocol 1 *discovers* the determinants; here we *use* them to predict and to attach a group-level importance ranking. We also show **honest, light evaluation hooks** (cross-validated metrics, a trivial baseline, per-protein probabilities) and keep heavy validation (repeated-CV, bootstrap CIs, shuffled-label controls) for **Protocol 9 — Validate**.

This protocol follows the shared 7-field protocol structure (*When to use it · Input · Run · Output · How to interpret · Common mistakes · Next step*) and belongs to the Protocols catalogue (epic #35).

## When to use it

Use this protocol when you have **two labelled sets of sequences** and want a model that not only predicts a new protein but tells you **why**. In glossary terms this is moving from *determinant discovery* to *prediction* (here, classification): a `TreeModel` rides on interpretable CPP features instead of a black box.

The biological question here: *given a transmembrane protein, is it a γ-secretase (GSEC) substrate, and which physicochemical properties of its TMD and juxtamembrane flanks drive that call?* Because every feature is a position-resolved physicochemical property (an AAontology scale evaluated at a sequence part and split), the fitted model's importances name the determinants — they are not anonymous weights.

If confirmed negatives are scarce (you mostly have positives plus unlabelled proteins), see the **PU note** near the end: `dPULearn` carves *reliable negatives* before you train.

## Input

The same `df_seq` as **Protocol 1**: one row per protein, a binary `label` column (test class = 1 = substrate, reference class = 0 = non-substrate), and `tmd_start` / `tmd_stop` boundaries from which the **TMD-centric parts** are derived. The default `df_parts` uses the composite parts `tmd`, `jmd_n_tmd_n`, and `tmd_c_jmd_c` (the TMD plus its two JMD-flank composites).

The CPP signature `df_feat` itself comes from **Protocol 1 — CPP signature**. To keep this protocol self-contained we re-run the minimal CPP path here on the bundled `DOM_GSEC` dataset.

In [1]:
import aaanalysis as aa

aa.options["verbose"] = False
aa.options["random_state"] = 42

# Two labelled sets of sequences (label: 1 = substrate/test, 0 = non-substrate/reference)
df_seq = aa.load_dataset(name="DOM_GSEC", n=25)   # n per class -> 50 proteins (2N rows)
labels = df_seq["label"].to_list()
df_seq.head()

,entry,sequence,label,tmd_start,tmd_stop,jmd_n,tmd,jmd_c
0,Q14802,MQKVTLGLLVFLAGFPVLDANDLEDKNSPFYYDWHSLQVGGLICAG...,0,37,59,NSPFYYDWHS,LQVGGLICAGVLCAMGIIIVMSA,KCKCKFGQKS
1,Q86UE4,MAARSWQDELAQQAEEGSARLREMLSVGLGFLRTELGLDLGLEPKR...,0,50,72,LGLEPKRYPG,WVILVGTGALGLLLLFLLGYGWA,AACAGARKKR
2,Q969W9,MHRLMGVNSTAAAAAGQPNVSCTCNCKRSLFQSMEITELEFVQIII...,0,41,63,FQSMEITELE,FVQIIIIVVVMMVMVVVITCLLS,HYKLSARSFI
3,P53801,MAPGVARGPTPYWRLRLGGAALLLLLIPVAAAQEPPGAACSQNTNK...,0,97,119,RWGVCWVNFE,ALIITMSVVGGTLLLGIAICCCC,CCRRKRSRKP
4,Q8IUW5,MAPRALPGSAVLAAAVFVGGAVSSPLVAPDNGSSRTLHSRTETTPS...,0,59,81,NDTGNGHPEY,IAYALVPVFFIMGLFGVLICHLL,KKKGYRCTTE


## Run

The **real** minimal path (not a one-liner):

1. Split each sequence into *parts* with `SequenceFeature.get_df_parts`.
2. Mine the discriminant signature with `CPP(df_parts=...).run(labels=...)` — `CPP` takes `df_parts`, **not** `df_seq` / `labels` directly.
3. Build the numerical feature matrix `X` with `SequenceFeature.feature_matrix`.
4. Fit the classifier with `TreeModel.fit(X, labels=...)` and attach the importance ranking with `add_feat_importance`.

In [2]:
# 1) Split each sequence into parts; default df_parts columns are
#    tmd, jmd_n_tmd_n, tmd_c_jmd_c (TMD plus the two JMD-flank composites).
sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)

# 2) Mine the most discriminant features (the CPP signature).
#    n_jobs=1 keeps it serial (multiprocessing spawn is fragile on
#    Python 3.14 + macOS without a __main__ guard).
cpp = aa.CPP(df_parts=df_parts)
df_feat = cpp.run(labels=labels, n_filter=50, n_jobs=1)
df_feat[["feature", "category", "subcategory", "abs_auc", "mean_dif"]].head()

,feature,category,subcategory,abs_auc,mean_dif
0,"TMD_C_JMD_C-Pattern(N,4,8,12)-CRAJ730103",Conformation,β-turn,0.426,-0.266
1,"TMD_C_JMD_C-Pattern(N,4,8,12)-BEGF750101",Conformation,α-helix,0.419,0.225
2,"TMD_C_JMD_C-Pattern(C,10,14)-CRAJ730103",Conformation,β-turn,0.408,-0.312
3,"TMD-Pattern(C,4,8,11)-BEGF750101",Conformation,α-helix,0.396,0.238
4,"TMD_C_JMD_C-Pattern(C,8,11,14)-ZIMJ680104",Energy,Isoelectric point,0.391,0.129


In [3]:
# 3) Build the feature matrix X and fit the interpretable classifier.
#    feature_matrix turns each PART-SPLIT-SCALE feature into one averaged
#    numeric value per protein -> X has shape (n_proteins, n_features).
X = sf.feature_matrix(features=df_feat["feature"], df_parts=df_parts, n_jobs=1)

# TreeModel aggregates several tree models over several rounds; .fit sets
# feat_importance, is_selected_ and list_models_ (defaults: n_rounds=5, no RFE).
tm = aa.TreeModel(verbose=False, random_state=42)
tm = tm.fit(X, labels=labels)

# Attach the Monte-Carlo feature importance (percent) as a column.
df_feat = tm.add_feat_importance(df_feat=df_feat)
df_feat[["feature", "subcategory", "abs_auc", "mean_dif", "feat_importance"]].head()

,feature,subcategory,abs_auc,mean_dif,feat_importance
0,"TMD_C_JMD_C-Pattern(N,4,8,12)-CRAJ730103",β-turn,0.426,-0.266,4.074
1,"TMD_C_JMD_C-Pattern(N,4,8,12)-BEGF750101",α-helix,0.419,0.225,5.689
2,"TMD_C_JMD_C-Pattern(C,10,14)-CRAJ730103",β-turn,0.408,-0.312,3.697
3,"TMD-Pattern(C,4,8,11)-BEGF750101",α-helix,0.396,0.238,1.910
4,"TMD_C_JMD_C-Pattern(C,8,11,14)-ZIMJ680104",Isoelectric point,0.391,0.129,0.829


### Honest evaluation hooks (light)

Three quick, honest checks — enough to know the model is non-trivial, **not** a substitute for validation:

1. **Cross-validated metrics** via `TreeModel.eval` over the full fitted feature set (`fit` ran with `use_rfe=False`, so no RFE selection was applied — all features are kept).
2. **A trivial baseline** — a majority-class `DummyClassifier`; a real model must beat its ~0.5 balanced accuracy.
3. **Per-protein probabilities** via `TreeModel.predict_proba` (a mean score and its Monte-Carlo std).

**Read these numbers as an optimistic upper bound, not an unbiased estimate.** The CPP signature was mined with `labels` on the *whole* 50-protein dataset, then this CV reuses exactly those globally-selected feature columns — so each fold's features were chosen using proteins that also sit in that fold's test split (feature-selection leakage). It is a sanity check / ceiling, not a generalization estimate. Leak-free nested or hold-out evaluation plus shuffled-label controls live in **Protocol 9 — Validate**.

To actually *reduce* the feature set (recursive feature elimination) rather than keep all features, fit with `TreeModel(...).fit(X, labels, use_rfe=True)` or call `select_features`; `is_selected_` then marks the retained subset.

In [4]:
# Cross-validated performance of the full fitted feature set (use_rfe=False
# keeps all features; is_selected_ is all-True). Optimistic: the CPP features
# were selected on the full dataset, so this is an upper bound, not a clean
# generalization estimate (see Protocol 9 for leak-free validation).
df_eval = tm.eval(X,
                  labels=labels,
                  list_is_selected=[tm.is_selected_],
                  list_metrics=["accuracy", "balanced_accuracy", "f1", "roc_auc"],
                  n_cv=5)

# Trivial majority-class baseline for comparison (~0.5 balanced accuracy).
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_score

baseline_bacc = cross_val_score(
    DummyClassifier(strategy="most_frequent"),
    X, labels, cv=5, scoring="balanced_accuracy").mean()

df_eval.assign(baseline_balanced_accuracy=round(float(baseline_bacc), 3))

,name,accuracy,balanced_accuracy,f1,roc_auc,baseline_balanced_accuracy
0,Set 1,0.92,0.92,0.9253,0.984,0.5


In [5]:
# Per-protein predicted probability of being a substrate (requires prior fit).
import pandas as pd

pred, pred_std = tm.predict_proba(X)
df_pred = pd.DataFrame({
    "entry": df_seq["entry"].to_list(),
    "label": labels,
    "p_substrate": pred.round(3),
    "p_std": pred_std.round(3),
})
# These are IN-SAMPLE (training) predictions, shown only to demonstrate the
# output shape -- they say nothing about generalization. p_std is the spread
# across the ensemble (all tree models over all rounds) (agreement), NOT held-out confidence: a small
# p_std means the models concur, not that the call is correct on unseen data.
# Score real, unseen proteins to judge generalization (see Protocol 9).
df_pred.sort_values("p_substrate", ascending=False).head()

,entry,label,p_substrate,p_std
33,P19022,1,1.0,0.0
29,Q06481,1,1.0,0.0
43,Q9ERC8,1,1.0,0.0
32,P09803,1,1.0,0.0
35,P09603,1,1.0,0.0


## Output

- A **fitted `TreeModel`** carrying `feat_importance` / `feat_importance_std` (group-level importance, percent), `is_selected_` (per-round feature selection) and `list_models_` (the fitted estimators).
- An **enriched `df_feat`** — the CPP signature plus the `feat_importance` column, so each determinant is ranked.
- A **`df_eval`** row of cross-validated metric means (and the trivial baseline) for the selected feature set.
- **Per-protein substrate probabilities** with Monte-Carlo std.

This feeds **Protocol 8 — Interpretability** (explain a single prediction) and **Protocol 9 — Validate** (trust the result).

## How to interpret

| Output | Non-expert reading |
| --- | --- |
| `balanced_accuracy` clearly above the ~0.5 baseline | the selected TMD/JMD physicochemistry is *plausibly* discriminative — confirm with leak-free validation in **Protocol 9** (this light CV is an optimistic upper bound) |
| high `roc_auc` | the model ranks substrates above non-substrates reliably |
| high `feat_importance` | the determinant the model **leans on most** for the call |
| sign of `mean_dif` for that feature | **direction** of the effect (positive = property higher in substrates) |
| `p_substrate` near 1 (or 0) | a substrate (or non-substrate) call; in-sample here, so judge confidence only on unseen proteins |
| large `p_std` | an unstable prediction — the models disagree |

`feat_importance` is **unsigned** (magnitude of influence only). Always pair it with `mean_dif` from the CPP signature to recover the biological direction — e.g. *higher side-chain length in the TMD core* rather than just *side-chain length matters*.

## Common mistakes

- **`CPP(df_seq=...)` / `CPP().run(df_seq, labels)`** — `CPP` takes `df_parts`; build them with `SequenceFeature.get_df_parts` first.
- **Skipping the feature matrix** — `TreeModel.fit` needs numeric `X` from `SequenceFeature.feature_matrix`, not `df_feat` or `df_seq`.
- **Reading `feat_importance` as signed** — it is magnitude only; combine with `mean_dif` for direction.
- **Calling `predict_proba` or `add_feat_importance` before `fit`** — both rely on attributes set during `.fit`.
- **Mistaking light eval for validation** — the CV row and baseline here are sanity checks; rigorous trust-building is **Protocol 9**.
- **Mis-using `dPULearn`** — it expects labels **1 (positive) and 2 (unlabelled)**, not 0/1, and `n_unl_to_neg` must be **below** the number of unlabelled samples.

## PU note — when confirmed negatives are scarce

Real GSEC datasets often have **confirmed substrates (positives, label 1)** but few confirmed non-substrates; the rest are **unlabelled (label 2)**. You cannot train a clean binary classifier on positives-plus-unlabelled directly.

`dPULearn` (core, no pro dependency) solves the first half: from the unlabelled pool it identifies **reliable negatives (label 0)** — the proteins most dissimilar from the positives in CPP feature space — which you can then feed to the classifier above.

Because CPP feature identifiers are **dataset-independent strings**, the *same* `df_feat["feature"]` builds the feature matrix for the PU proteins. We demonstrate on the bundled `DOM_GSEC_PU` dataset.

In [6]:
# Positives (1) + unlabelled (2): n=20 -> 20 positives and 20 unlabelled.
df_pu = aa.load_dataset(name="DOM_GSEC_PU", n=20)
labels_pu = df_pu["label"].to_list()

# Reuse the SAME CPP feature ids to build X for the PU proteins.
df_parts_pu = sf.get_df_parts(df_seq=df_pu)
X_pu = sf.feature_matrix(features=df_feat["feature"], df_parts=df_parts_pu, n_jobs=1)

# Carve reliable negatives (0) from the unlabelled pool (2).
# n_unl_to_neg must be >= 1 and < the number of unlabelled samples (20 here).
dpul = aa.dPULearn(verbose=False, random_state=42)
dpul = dpul.fit(X=X_pu, labels=labels_pu, n_unl_to_neg=10)

# labels_: 1 = positive, 0 = newly identified reliable negative, 2 = still unlabelled.
import collections
collections.Counter(dpul.labels_.tolist())

Counter({1: 20, 2: 10, 0: 10})

## Next step

- **Explain one prediction** — turn the classifier into per-sample, single-residue explanations: see **Protocol 8 — Interpretability** (`ShapModel`, sample-level `CPPPlot.feature_map`).
- **Trust the result** — repeated-CV, bootstrap confidence intervals and shuffled-label controls: see **Protocol 9 — Validate**.